# Part 3 — FinBERT Sentiment Analysis

See `README.md` for project context and motivation.

This notebook applies **FinBERT** (Araci 2019; `ProsusAI/finbert`) to each
earnings call transcript to extract document-level sentiment scores
(positive / negative / neutral with probabilities). The results are then
merged with the LDA topic assignments and geoeconomic risk flags from
Parts 1 and 2 to answer:

- Which LDA topics are discussed with negative vs positive tone?
- Does sentiment shift across reporting quarters?
- Do executives speak more positively than analysts?
- Are geoeconomically-flagged transcripts more negatively toned?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Setup complete.')

## 1. Load FinBERT

FinBERT is a BERT-base model fine-tuned on ~10,000 financial news sentences
(Reuters/Bloomberg) and the Financial PhraseBank. It classifies text as
**positive**, **negative**, or **neutral** with calibrated probabilities.

Because BERT has a hard limit of 512 tokens, transcripts are split into
non-overlapping 400-token chunks; scores are averaged across chunks.

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)

LABELS = ['positive', 'negative', 'neutral']   # FinBERT label order
MAX_CHUNK_TOKENS = 400
BATCH_SIZE       = 16

print(f'Loaded {MODEL_NAME} — label order: {LABELS}')

## 2. Inference helpers

In [ ]:
def chunk_text(text: str, max_tokens: int = MAX_CHUNK_TOKENS) -> list:
    """
    Split text into overlapping-free chunks of at most max_tokens tokens.
    Works at the sentence boundary where possible, otherwise at the token level.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    # Split into sentences first, then bin them into chunks
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_len = [], [], 0
    for s in sentences:
        n = len(tokenizer.tokenize(s))
        if current_len + n > max_tokens and current:
            chunks.append(' '.join(current))
            current, current_len = [], 0
        current.append(s)
        current_len += n
    if current:
        chunks.append(' '.join(current))
    return chunks


def score_chunks(chunks: list) -> np.ndarray:
    """
    Run FinBERT on a list of text chunks and return mean probability vector
    [p_positive, p_negative, p_neutral] averaged across all chunks.
    Returns [NaN, NaN, NaN] for empty input.
    """
    if not chunks:
        return np.array([np.nan, np.nan, np.nan])
    probs_list = []
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        enc   = tokenizer(batch, return_tensors='pt', truncation=True,
                          max_length=512, padding=True).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = softmax(logits, dim=-1).cpu().numpy()   # shape (batch, 3)
        probs_list.append(probs)
    all_probs = np.vstack(probs_list)                  # (n_chunks, 3)
    return all_probs.mean(axis=0)


def score_text(text: str) -> dict:
    """Return FinBERT sentiment scores for a single text string."""
    chunks = chunk_text(text)
    probs  = score_chunks(chunks)
    return {
        'p_positive': float(probs[0]),
        'p_negative': float(probs[1]),
        'p_neutral':  float(probs[2]),
        'sentiment':  LABELS[int(np.argmax(probs))],
        'n_chunks':   len(chunks),
    }


# Quick sanity check
t1 = score_text('Revenue beat estimates and earnings per share rose sharply.')
t2 = score_text('Tariff escalation and supply chain disruptions weighed on margins.')
print('Positive example:', t1)
print('Negative example:', t2)

## 3. Run inference on the corpus

We score three text fields per document:
- `full_text` — overall transcript sentiment
- `exec_text` — executive (management) speech only
- `analyst_text` — analyst questions only

Results are cached to `data/finbert_sentiment.csv` so re-runs are instant.

In [ ]:
CACHE_PATH = Path('data/finbert_sentiment.csv')

corpus = pd.read_csv('data/corpus_documents.csv')
print(f'Corpus: {len(corpus):,} documents')

if CACHE_PATH.exists():
    print('Cache found — loading from disk.')
    sent_df = pd.read_csv(CACHE_PATH)
else:
    print('Running FinBERT inference (this will take a while on CPU) …')
    records = []
    for i, row in corpus.iterrows():
        if (i + 1) % 100 == 0:
            print(f'  {i+1:>4}/{len(corpus)}')
        rec = {'url': row['url']}
        for field in ('full_text', 'exec_text', 'analyst_text'):
            scores = score_text(str(row.get(field) or ''))
            for k, v in scores.items():
                rec[f'{field}__{k}'] = v
        records.append(rec)
    sent_df = pd.DataFrame(records)
    sent_df.to_csv(CACHE_PATH, index=False)
    print(f'Saved {CACHE_PATH}')

print(f'Loaded sentiment data: {len(sent_df):,} rows, {len(sent_df.columns)} columns')

## 4. Merge with metadata, LDA topics, and geoeconomic flags

In [ ]:
meta_cols = ['url', 'company_name', 'ticker', 'reporting_period', 'event_type']
meta = corpus[meta_cols].copy()

# LDA dominant topic
gamma = pd.read_csv('data/doc_topic_distributions.csv',
                    usecols=['url', 'dominant_topic', 'dominant_label'])

# Geoeconomic risk flags
geo = pd.read_csv('data/geoeconomic_matches.csv',
                  usecols=['url', 'trade_risk', 'sanctions_risk',
                           'embargo_risk', 'geopolitical_risk', 'any_georisk'])

df = (
    sent_df
    .merge(meta,  on='url', how='left')
    .merge(gamma, on='url', how='left')
    .merge(geo,   on='url', how='left')
)

# Convenience: net sentiment = p_positive − p_negative  (range −1 to +1)
df['net_sentiment'] = df['full_text__p_positive'] - df['full_text__p_negative']
df['exec_net']      = df['exec_text__p_positive'] - df['exec_text__p_negative']
df['analyst_net']   = df['analyst_text__p_positive'] - df['analyst_text__p_negative']

PERIOD_ORDER = ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025',
                'Q1 2026', 'Q2 2026', 'Q3 2026', 'Q4 2026']
df['reporting_period'] = pd.Categorical(
    df['reporting_period'], categories=PERIOD_ORDER, ordered=True
)

print(df[['reporting_period', 'dominant_label', 'full_text__sentiment',
          'net_sentiment', 'any_georisk']].describe(include='all').T)

## 5. Visualisations

In [ ]:
# ── Fig 1: Overall sentiment distribution ──────────────────────────────────

SENT_COLORS = {'positive': '#2166AC', 'neutral': '#AAAAAA', 'negative': '#D6604D'}

counts = df['full_text__sentiment'].value_counts().reindex(
    ['positive', 'neutral', 'negative'])
total  = counts.sum()

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(counts.index, counts.values,
              color=[SENT_COLORS[s] for s in counts.index],
              alpha=0.88, edgecolor='white', linewidth=1.0, width=0.55)

for bar, (label, count) in zip(bars, counts.items()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + total * 0.008,
            f'{count:,}\n({100*count/total:.1f}%)',
            ha='center', va='bottom', fontsize=10.5, color='#333333',
            linespacing=1.4)

ax.set_ylabel('Number of transcripts', fontsize=11, labelpad=8)
ax.set_title('Overall Sentiment Distribution (FinBERT, full transcript)',
             fontsize=13, fontweight='bold', pad=12, loc='left')
ax.set_ylim(0, counts.max() * 1.25)
ax.yaxis.grid(True, linestyle='--', alpha=0.35, color='grey', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
plt.tight_layout(pad=1.5)
plt.savefig('figures/14_finbert_sentiment_dist.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved figures/14_finbert_sentiment_dist.png')

In [ ]:
# ── Fig 2: Net sentiment by LDA topic ──────────────────────────────────────
# Net sentiment = p_positive - p_negative; higher = more positive tone.

topic_sent = (
    df.dropna(subset=['dominant_label', 'net_sentiment'])
    .groupby('dominant_label')
    .agg(mean_net=('net_sentiment', 'mean'),
         n=('net_sentiment', 'count'))
    .sort_values('mean_net')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, max(5, len(topic_sent) * 0.42 + 1.5)))
colors = ['#D6604D' if v < 0 else '#2166AC' for v in topic_sent['mean_net']]
bars   = ax.barh(topic_sent['dominant_label'], topic_sent['mean_net'],
                 color=colors, alpha=0.88, edgecolor='white',
                 linewidth=0.7, height=0.7)

for bar, (_, row) in zip(bars, topic_sent.iterrows()):
    x_pos = row['mean_net'] + (0.003 if row['mean_net'] >= 0 else -0.003)
    ha    = 'left' if row['mean_net'] >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"n={row['n']:,}", va='center', ha=ha,
            fontsize=8, color='#555555')

ax.axvline(0, color='#888888', linewidth=1.2, linestyle='--')
ax.set_xlabel('Mean net sentiment (p_positive − p_negative)', fontsize=11,
              labelpad=8)
ax.set_title('Mean Net Sentiment by LDA Topic',
             fontsize=13, fontweight='bold', pad=12, loc='left')
ax.xaxis.grid(True, linestyle='--', alpha=0.35, color='grey', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
plt.tight_layout(pad=1.5)
plt.savefig('figures/15_finbert_by_topic.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved figures/15_finbert_by_topic.png')

In [ ]:
# ── Fig 3: Net sentiment by reporting period ────────────────────────────────

period_sent = (
    df.dropna(subset=['reporting_period', 'net_sentiment'])
    .groupby('reporting_period', observed=True)
    .agg(mean_net=('net_sentiment', 'mean'),
         n=('net_sentiment', 'count'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(period_sent))
colors = ['#D6604D' if v < 0 else '#2166AC' for v in period_sent['mean_net']]
bars   = ax.bar(x, period_sent['mean_net'], color=colors, alpha=0.88,
                edgecolor='white', linewidth=0.8, width=0.60)

for xi, (_, row) in enumerate(period_sent.iterrows()):
    y_off = 0.002 if row['mean_net'] >= 0 else -0.002
    va    = 'bottom' if row['mean_net'] >= 0 else 'top'
    ax.text(xi, row['mean_net'] + y_off,
            f"n={row['n']:,}", ha='center', va=va,
            fontsize=8.5, color='#555555')

ax.axhline(0, color='#888888', linewidth=1.2, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(period_sent['reporting_period'].astype(str),
                   rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Mean net sentiment', fontsize=11, labelpad=8)
ax.set_title('Mean Net Sentiment by Reporting Period',
             fontsize=13, fontweight='bold', pad=12, loc='left')
ax.set_xlabel(
    'Note: corpus dominated by Q4 2025. Cross-quarter comparisons are illustrative.',
    fontsize=8.5, color='#888888', labelpad=10, style='italic'
)
ax.yaxis.grid(True, linestyle='--', alpha=0.35, color='grey', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
plt.tight_layout(pad=1.5)
plt.savefig('figures/16_finbert_by_period.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved figures/16_finbert_by_period.png')

In [ ]:
# ── Fig 4: Executive vs analyst net sentiment ───────────────────────────────

role_data = pd.DataFrame({
    'Role':         ['Executives', 'Analysts'],
    'mean_net':     [df['exec_net'].mean(), df['analyst_net'].mean()],
    'std':          [df['exec_net'].std(),  df['analyst_net'].std()],
    'n':            [df['exec_net'].notna().sum(), df['analyst_net'].notna().sum()],
})
role_data['se'] = role_data['std'] / np.sqrt(role_data['n'])

fig, ax = plt.subplots(figsize=(6, 4.5))
ROLE_COLORS = ['#2166AC', '#D6604D']
bars = ax.bar(role_data['Role'], role_data['mean_net'],
              color=ROLE_COLORS, alpha=0.88, edgecolor='white',
              linewidth=1.0, width=0.45,
              yerr=role_data['se'] * 1.96, capsize=5,
              error_kw={'elinewidth': 1.5, 'ecolor': '#444444', 'capthick': 1.5})

for bar, (_, row) in zip(bars, role_data.iterrows()):
    y_off = 0.002 if row['mean_net'] >= 0 else -0.002
    ax.text(bar.get_x() + bar.get_width() / 2,
            row['mean_net'] + row['se'] * 1.96 + abs(y_off) * 3,
            f"{row['mean_net']:+.3f}\n(n={row['n']:,})",
            ha='center', va='bottom', fontsize=10, color='#333333',
            linespacing=1.3)

ax.axhline(0, color='#888888', linewidth=1.2, linestyle='--')
ax.set_ylabel('Mean net sentiment (p_pos − p_neg)', fontsize=11, labelpad=8)
ax.set_title('Executive vs Analyst Net Sentiment',
             fontsize=13, fontweight='bold', pad=12, loc='left')
ax.set_subtitle = None
ax.yaxis.grid(True, linestyle='--', alpha=0.35, color='grey', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
plt.tight_layout(pad=1.5)
plt.savefig('figures/17_finbert_exec_vs_analyst.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved figures/17_finbert_exec_vs_analyst.png')

In [ ]:
# ── Fig 5: Sentiment — geoeconomic-flagged vs non-flagged ──────────────────

geo_sent = (
    df.dropna(subset=['any_georisk', 'net_sentiment'])
    .groupby('any_georisk')
    .agg(mean_net=('net_sentiment', 'mean'),
         std=('net_sentiment', 'std'),
         n=('net_sentiment', 'count'))
    .reset_index()
)
geo_sent['label'] = geo_sent['any_georisk'].map(
    {True: 'Geoeconomic\nrisk flagged', False: 'Not flagged'})
geo_sent['se'] = geo_sent['std'] / np.sqrt(geo_sent['n'])

fig, ax = plt.subplots(figsize=(6, 4.5))
GEO_COLORS = ['#D6604D', '#2166AC']
bars = ax.bar(geo_sent['label'], geo_sent['mean_net'],
              color=GEO_COLORS, alpha=0.88, edgecolor='white',
              linewidth=1.0, width=0.45,
              yerr=geo_sent['se'] * 1.96, capsize=5,
              error_kw={'elinewidth': 1.5, 'ecolor': '#444444', 'capthick': 1.5})

for bar, (_, row) in zip(bars, geo_sent.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            row['mean_net'] + row['se'] * 1.96 + 0.003,
            f"{row['mean_net']:+.3f}\n(n={row['n']:,})",
            ha='center', va='bottom', fontsize=10, color='#333333',
            linespacing=1.3)

ax.axhline(0, color='#888888', linewidth=1.2, linestyle='--')
ax.set_ylabel('Mean net sentiment (p_pos − p_neg)', fontsize=11, labelpad=8)
ax.set_title('Sentiment: Geoeconomic-Flagged vs Not Flagged',
             fontsize=13, fontweight='bold', pad=12, loc='left')
ax.yaxis.grid(True, linestyle='--', alpha=0.35, color='grey', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
plt.tight_layout(pad=1.5)
plt.savefig('figures/18_finbert_georisk_vs_not.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved figures/18_finbert_georisk_vs_not.png')

In [ ]:
# Export merged results
out_cols = [
    'url', 'company_name', 'ticker', 'reporting_period', 'event_type',
    'dominant_label',
    'full_text__p_positive', 'full_text__p_negative', 'full_text__p_neutral',
    'full_text__sentiment', 'net_sentiment',
    'exec_net', 'analyst_net',
    'trade_risk', 'sanctions_risk', 'embargo_risk',
    'geopolitical_risk', 'any_georisk',
]
out_cols = [c for c in out_cols if c in df.columns]
df[out_cols].to_csv('data/finbert_results.csv', index=False)
print(f'Saved data/finbert_results.csv  ({len(df):,} rows)')

## Summary

| File | Description |
|---|---|
| `data/finbert_sentiment.csv` | Raw FinBERT scores per document (all three text fields) |
| `data/finbert_results.csv` | Merged: scores + LDA topics + geoeconomic flags |
| `figures/14_finbert_sentiment_dist.png` | Overall sentiment distribution |
| `figures/15_finbert_by_topic.png` | Net sentiment by LDA topic |
| `figures/16_finbert_by_period.png` | Net sentiment by reporting quarter |
| `figures/17_finbert_exec_vs_analyst.png` | Executive vs analyst sentiment |
| `figures/18_finbert_georisk_vs_not.png` | Geoeconomic-flagged vs non-flagged sentiment |